# E3/E4 — đánh giá VALID và TEST sau 30 epoch trên Kaggle GPU
Gắn Input: Output E3 (results_e3.zip), Output E4 (results_e4.zip) và dataset ROD Final A4 gốc dùng khi train. Hỗ trợ cả ZIP nguyên và Output đã được Kaggle giải nén. Bật GPU và Internet để cài Ultralytics. Không cần HF token.

Chạy Save Version > Save & Run All. Notebook kiểm tra checkpoint 30/30 và SHA256, chọn ngưỡng trên 3318 ảnh VALID, khóa cấu hình rồi đánh giá 1611 ảnh TEST. Không train lại. Có log tiến độ và cache để chạy lại trong cùng runtime; cache trong /kaggle/working không tự tồn tại qua phiên mới.

Kết quả: /kaggle/working/evaluation_30epochs/summary_E3_E4.csv và ZIP evaluation_30epochs_results.zip. Dataset phải giữ data.yaml gốc để kiểm SHA256; đường dẫn Windows trong YAML không ảnh hưởng vì runner đọc trực tiếp thư mục split.


In [ ]:
import os, sys, subprocess
os.environ['YOLO_AUTOINSTALL'] = 'false'
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet', '--no-deps',
                       'ultralytics==8.4.165', 'ultralytics-thop'])
import torch
if not torch.cuda.is_available():
    raise RuntimeError('Enable Kaggle Settings > Accelerator > GPU before running')
print('GPU:', torch.cuda.get_device_name(0), '; torch:', torch.__version__, flush=True)


In [ ]:
from pathlib import Path
import sys
CODE_ROOT = Path('/kaggle/working/rod_eval_code')
SOURCES = {'src/kaggle_hf_shared.py': '"""Embedded in both Kaggle notebooks. No local import is required at runtime."""\n\nimport hashlib\nimport json\nimport os\nimport re\nimport shutil\nimport time\nimport uuid\nfrom pathlib import Path\n\n\nHF_REPO_ID = "iraprp/deeplearning"\nHF_SECRET_NAME = "HF_TOKEN2"\n\n\ndef sha256_file(path):\n    h = hashlib.sha256()\n    with Path(path).open("rb") as stream:\n        for part in iter(lambda: stream.read(4 * 1024 * 1024), b""):\n            h.update(part)\n    return h.hexdigest()\n\n\ndef fingerprint(value):\n    return hashlib.sha256(json.dumps(value, sort_keys=True, ensure_ascii=False, separators=(",", ":")).encode("utf-8")).hexdigest()\n\n\ndef atomic_json(path, value):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    temp = path.with_suffix(path.suffix + ".tmp")\n    temp.write_text(json.dumps(value, indent=2, ensure_ascii=False, sort_keys=True), encoding="utf-8")\n    os.replace(temp, path)\n\n\ndef validate_progress(manifest, folder, model, config_hash, dataset_hash, max_epoch=10):\n    if manifest.get("schema") != 2 or manifest.get("folder") != folder or manifest.get("model") != model:\n        raise RuntimeError("HF manifest thuộc mô hình/folder khác hoặc sai schema")\n    if manifest.get("config_sha256") != config_hash or manifest.get("dataset_sha256") != dataset_hash:\n        raise RuntimeError("HF checkpoint khác config/dataset; dừng, không train lại")\n    epoch = manifest.get("completed_epoch")\n    if type(epoch) is not int or not 0 <= epoch <= max_epoch:\n        raise RuntimeError("HF completed_epoch không hợp lệ")\n    if manifest.get("is_complete") is not (epoch == max_epoch):\n        raise RuntimeError("HF trạng thái COMPLETE không khớp epoch")\n    if not isinstance(manifest.get("sha256"), dict) or not manifest["sha256"]:\n        raise RuntimeError("HF manifest thiếu checksum")\n    return epoch\n\n\nclass HFStore:\n    """Immutable snapshot first, latest pointer second. Failed upload cannot advance latest."""\n\n    def __init__(self, api, token, repo_type, folder, model, config_hash, dataset_hash, work, max_epoch=10, repo_id=None, compact=False):\n        if folder not in ("YOLO 11s", "Faster R-CNN", "E3_30epochs", "E4_30epochs", "E6_30epochs"):\n            raise ValueError("Unknown experiment folder")\n        if type(max_epoch) is not int or max_epoch not in (10, 30):\n            raise ValueError("Invalid epoch budget")\n        self.max_epoch = max_epoch\n        self.compact = compact\n        self.repo_id = HF_REPO_ID if repo_id is None else repo_id\n        self.api, self.token, self.repo_type = api, token, repo_type\n        self.folder, self.model = folder, model\n        self.config_hash, self.dataset_hash = config_hash, dataset_hash\n        self.work = Path(work)\n        self.work.mkdir(parents=True, exist_ok=True)\n\n    def _files(self):\n        return set(self.api.list_repo_files(repo_id=self.repo_id, repo_type=self.repo_type, token=self.token))\n\n    def _download(self, remote):\n        from huggingface_hub import hf_hub_download\n        return Path(hf_hub_download(repo_id=self.repo_id, repo_type=self.repo_type,\n                                    filename=remote, token=self.token, local_dir=str(self.work / "download"),\n                                    force_download=remote.endswith("latest.json")))\n\n    def restore(self):\n        files = self._files()  # connection/auth errors must propagate\n        prefix = self.folder + "/"\n        relevant = [name for name in files if name.startswith(prefix)]\n        pointer_name = prefix + "latest.json"\n        if pointer_name not in files:\n            if relevant:\n                raise RuntimeError("HF folder có dữ liệu nhưng thiếu latest.json; dừng để kiểm tra thủ công")\n            return None\n        pointer = json.loads(self._download(pointer_name).read_text(encoding="utf-8"))\n        snap = pointer.get("snapshot")\n        if not isinstance(snap, str) or not snap.startswith(prefix + "snapshots/") or ".." in snap:\n            raise RuntimeError("HF latest pointer không hợp lệ")\n        manifest_name = snap + "/manifest.json"\n        if manifest_name not in files:\n            raise RuntimeError("HF latest pointer trỏ tới snapshot thiếu manifest")\n        manifest_path = self._download(manifest_name)\n        if sha256_file(manifest_path) != pointer.get("manifest_sha256"):\n            raise RuntimeError("HF manifest SHA256 không khớp latest pointer")\n        manifest = json.loads(manifest_path.read_text(encoding="utf-8"))\n        validate_progress(manifest, self.folder, self.model, self.config_hash, self.dataset_hash, self.max_epoch)\n        if manifest.get("snapshot") != snap:\n            raise RuntimeError("HF snapshot ID không nhất quán")\n        local = {}\n        for name, expected in manifest["sha256"].items():\n            if name != Path(name).name or snap + "/" + name not in files:\n                raise RuntimeError("HF snapshot thiếu artifact hoặc tên file không hợp lệ")\n            path = self._download(snap + "/" + name)\n            if sha256_file(path) != expected:\n                raise RuntimeError("HF artifact SHA256 không khớp: " + name)\n            local[name] = path\n        for alias, target in manifest.get("artifact_aliases", {}).items():\n            if (alias not in ("best.pt", "best_weights.pth") or target not in ("checkpoint.pt", "recovery_full.pt")\n                or target not in local or alias in local):\n                raise RuntimeError("HF artifact alias không hợp lệ")\n            local[alias] = local[target]\n        return manifest, local\n\n    def _compact_files(self, files):\n        """Keep the full resume payload; remove only redundant/unneeded uploads."""\n        recovery = "checkpoint.pt" if self.model == "fasterrcnn_resnet50_fpn_v2" else "recovery_full.pt"\n        required = ({"checkpoint.pt", "metrics.csv", "config.json", "environment.json"}\n                    if recovery == "checkpoint.pt" else\n                    {"recovery_full.pt", "results.csv", "args.yaml", "config.json", "environment.json"})\n        best = "best_weights.pth" if recovery == "checkpoint.pt" else "best.pt"\n        if not required.issubset(files):\n            raise RuntimeError("Compact upload thiếu file resume/config/metrics bắt buộc")\n        selected = {name: source for name, source in files.items() if name in required | {best}}\n        aliases = {}\n        if best in selected and sha256_file(selected[best]) == sha256_file(selected[recovery]):\n            aliases[best] = recovery\n            del selected[best]\n        return selected, aliases\n\n    def _prune_compact(self, current):\n        """Bound the live tree, never squash shared repo history during training.\n\n        Only compact-v1 snapshots of this exact folder/config/dataset are owned.\n        Unknown snapshots, source repositories, other models, and future progress\n        are never deleted. A conditional commit fails if another writer commits.\n        """\n        if self.repo_id != HF_REPO_ID:\n            raise RuntimeError("Refusing retention writes to source repository")\n        head = self.api.repo_info(repo_id=self.repo_id, repo_type=self.repo_type, token=self.token).sha\n        files = set(self.api.list_repo_files(repo_id=self.repo_id, repo_type=self.repo_type,\n                                             token=self.token, revision=head))\n        pointer_name = self.folder + "/latest.json"\n        pointer = json.loads(self._download(pointer_name).read_text(encoding="utf-8"))\n        if (pointer.get("snapshot") != current["snapshot"] or\n            pointer.get("manifest_sha256") != sha256_file(self._download(current["snapshot"] + "/manifest.json"))):\n            raise RuntimeError("Latest changed; retention skipped")\n        prefix = self.folder + "/snapshots/"\n        records = {}\n        for name in sorted(files):\n            if not name.startswith(prefix) or not name.endswith("/manifest.json"):\n                continue\n            snap = name.rsplit("/", 1)[0]\n            if snap.count("/") != 2:\n                continue\n            m = json.loads(self._download(name).read_text(encoding="utf-8"))\n            if (m.get("storage_policy") != "compact-v1" or m.get("folder") != self.folder or\n                m.get("snapshot") != snap or m.get("model") != self.model or\n                m.get("config_sha256") != self.config_hash or m.get("dataset_sha256") != self.dataset_hash):\n                continue\n            validate_progress(m, self.folder, self.model, self.config_hash, self.dataset_hash, self.max_epoch)\n            if type(m.get("next_sample", 0)) is not int or m.get("next_sample", 0) < 0:\n                raise RuntimeError("Invalid progress during retention")\n            actual = {p for p in files if p.startswith(snap + "/")}\n            expected = {snap + "/manifest.json"}\n            for artifact in m["sha256"]:\n                if artifact != Path(artifact).name or "/" in artifact or "\\\\" in artifact:\n                    raise RuntimeError("Unsafe artifact path during retention")\n                expected.add(snap + "/" + artifact)\n            if actual == expected:\n                records[snap] = m\n        if current["snapshot"] not in records:\n            raise RuntimeError("Current snapshot incomplete; no retention deletion")\n        def progress(s):\n            return records[s]["completed_epoch"], records[s].get("next_sample", 0)\n        keep = {current["snapshot"]}\n        for epoch in (10, 20, 30):\n            complete = [s for s, m in records.items() if m["completed_epoch"] == epoch and m.get("next_sample", 0) == 0]\n            if complete:\n                keep.add(max(complete, key=lambda s: (s == current["snapshot"], s)))\n        # Defensive fallback if an early/partial current snapshot has no best yet.\n        best = [s for s, m in records.items() if\n                {"best.pt", "best_weights.pth"} & (set(m["sha256"]) | set(m.get("artifact_aliases", {})))]\n        if best and not ({"best.pt", "best_weights.pth"} &\n                         (set(current["sha256"]) | set(current.get("artifact_aliases", {})))):\n            keep.add(max(best, key=lambda s: (progress(s), s)))\n        # Never prune a snapshot at more advanced progress, even if not latest yet.\n        obsolete = {s for s in records if s not in keep and progress(s) <= progress(current["snapshot"])}\n        paths = sorted(p for p in files if p.rsplit("/", 1)[0] in obsolete)\n        if not paths:\n            return\n        from huggingface_hub import CommitOperationDelete\n        self.api.create_commit(repo_id=self.repo_id, repo_type=self.repo_type, token=self.token,\n            parent_commit=head, revision="main", operations=[CommitOperationDelete(path_in_repo=p) for p in paths],\n            commit_message=f"{self.folder}: prune verified compact snapshots; keep latest/best/milestones")\n        print("HF retention:", self.folder, "removed snapshots=", len(obsolete),\n              "; retained latest/best/milestones. Git history unchanged.", flush=True)\n\n    def _retry(self, action):\n        for attempt in range(3):\n            try:\n                return action()\n            except Exception as exc:\n                response = getattr(exc, "response", None)\n                status = getattr(response, "status_code", None)\n                detail = getattr(exc, "server_message", None)\n                if not detail and response is not None:\n                    try:\n                        body = response.json()\n                        if isinstance(body, dict):\n                            detail = body.get("error") or body.get("message")\n                    except (ValueError, AttributeError):\n                        pass\n                # Some HF/Xet errors have only a plain-text HTTP response.\n                body_text = getattr(response, "text", "") if response is not None else ""\n                detail = str(detail or exc)\n                if body_text and str(body_text) not in detail:\n                    detail += "\\nResponse: " + str(body_text)\n                if self.token:\n                    detail = detail.replace(self.token, "[REDACTED]")\n                detail = re.sub(r"hf_[A-Za-z0-9_]+", "[REDACTED]", detail)\n                detail = re.sub(r"Bearer\\s+\\S+", "Bearer [REDACTED]", detail, flags=re.I)\n                detail = re.sub(r"https?://[^\\s]+", "[URL REDACTED]", detail)[:6000]\n                diagnostic = f"{type(exc).__name__}; HTTP {status}; {detail}"\n                record = {"time_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),\n                          "folder": self.folder, "operation": getattr(self, "_upload_operation", "upload"),\n                          "attempt": attempt + 1, "error_type": type(exc).__name__,\n                          "http_status": status, "server_message": detail,\n                          "local_checkpoint_retained": True}\n                # Keep diagnostics in Kaggle Output even when Save & Run All fails.\n                try:\n                    with (self.work / "hf_upload_errors.jsonl").open("a", encoding="utf-8") as log:\n                        log.write(json.dumps(record, ensure_ascii=False) + "\\n")\n                except OSError:\n                    print("Không ghi được log HF ra đĩa; xem lỗi bên dưới.", flush=True)\n                print("HF upload lỗi", attempt + 1, diagnostic, "; bản local còn ở", self.work, flush=True)\n                if attempt == 2:\n                    raise RuntimeError("HF upload thất bại; giữ Kaggle Output và KHÔNG đánh dấu đã đồng bộ. "\n                                       + diagnostic) from None\n                time.sleep(5 * (attempt + 1))\n\n    def save(self, files, completed_epoch, extra=None):\n        aliases = {}\n        if self.compact:\n            files, aliases = self._compact_files(files)\n        snap = self.folder + "/snapshots/" + uuid.uuid4().hex\n        stage = self.work / "snapshots" / snap.rsplit("/", 1)[-1]\n        stage.mkdir(parents=True, exist_ok=False)\n        hashes = {}\n        for name, source in files.items():\n            if name != Path(name).name or not Path(source).is_file():\n                raise RuntimeError("Artifact không hợp lệ: " + name)\n            target = stage / name\n            # Kaggle checkpoint and stage live on the same filesystem. Hardlinks\n            # avoid duplicating a large optimizer checkpoint during HF upload.\n            # Fall back to a copy if the filesystem does not support hardlinks.\n            try:\n                os.link(Path(source).resolve(), target)\n            except OSError:\n                shutil.copy2(source, target)\n            hashes[name] = sha256_file(target)\n        manifest = {**(extra or {}), "schema": 2, "folder": self.folder, "model": self.model,\n                    "snapshot": snap, "config_sha256": self.config_hash,\n                    "dataset_sha256": self.dataset_hash,\n                    "completed_epoch": int(completed_epoch), "is_complete": int(completed_epoch) == self.max_epoch,\n                    "sha256": hashes}\n        if self.compact:\n            manifest.update(storage_policy="compact-v1", artifact_aliases=aliases)\n        validate_progress(manifest, self.folder, self.model, self.config_hash, self.dataset_hash, self.max_epoch)\n        atomic_json(stage / "manifest.json", manifest)\n        self._upload_operation = "upload_folder"\n        print("HF upload_folder bắt đầu:", self.folder, "epoch", completed_epoch,\n              "repo=", self.repo_id, "repo_type=", self.repo_type, "remote=", snap,\n              "files=", len(files) + 1,\n              "upload_MiB=", round(sum(p.stat().st_size for p in stage.iterdir()) / 1024**2, 2),\n              "local=", stage, flush=True)\n        self._retry(lambda: self.api.upload_folder(folder_path=str(stage), path_in_repo=snap,\n                     repo_id=self.repo_id, repo_type=self.repo_type, token=self.token,\n                     commit_message=f"{self.folder}: snapshot epoch {completed_epoch}"))\n        # Verify server snapshot before moving pointer. A failed pointer commit leaves previous version valid.\n        remote_manifest = self._download(snap + "/manifest.json")\n        if sha256_file(remote_manifest) != sha256_file(stage / "manifest.json"):\n            raise RuntimeError("HF upload manifest verification failed")\n        pointer = self.work / "latest.json"\n        atomic_json(pointer, {"snapshot": snap, "manifest_sha256": sha256_file(stage / "manifest.json")})\n        self._upload_operation = "upload_latest_pointer"\n        print("HF snapshot manifest verified; updating latest:", self.folder, flush=True)\n        self._retry(lambda: self.api.upload_file(path_or_fileobj=str(pointer), path_in_repo=self.folder + "/latest.json",\n                    repo_id=self.repo_id, repo_type=self.repo_type, token=self.token,\n                    commit_message=f"{self.folder}: advance epoch {completed_epoch}"))\n        remote_pointer = json.loads(self._download(self.folder + "/latest.json").read_text(encoding="utf-8"))\n        if remote_pointer != json.loads(pointer.read_text(encoding="utf-8")):\n            raise RuntimeError("HF latest pointer chưa được xác nhận sau upload")\n        if self.compact:\n            remote_files = self._files()\n            if any(snap + "/" + name not in remote_files for name in hashes):\n                raise RuntimeError("Uploaded snapshot missing an artifact; retention NOT run")\n            try:\n                self._prune_compact(manifest)\n            except Exception as error:\n                # New recovery is durable; cleanup conflicts must not lose it or\n                # stop a successful epoch. Retry retention on the next save.\n                warning = {"folder": self.folder, "snapshot": snap, "retention": "SKIPPED",\n                           "error_type": type(error).__name__,\n                           "http_status": getattr(getattr(error, "response", None), "status_code", None)}\n                atomic_json(self.work / "retention_warning.json", warning)\n                print("WARNING: retention skipped; checkpoint uploaded. Extra snapshots kept:",\n                      type(error).__name__, flush=True)\n        # Only temporary staging files are removed; the source checkpoint in\n        # Kaggle Output and the verified immutable HF snapshot remain intact.\n        if stage.resolve().parent != (self.work / "snapshots").resolve():\n            raise RuntimeError("Local staging path outside expected snapshots directory")\n        for staged_file in stage.iterdir():\n            if not staged_file.is_file():\n                raise RuntimeError("Unexpected directory in local staging snapshot")\n            staged_file.unlink()\n        stage.rmdir()\n        print("HF snapshot verified:", self.folder, "epoch", completed_epoch,\n              "next_sample", manifest.get("next_sample", 0), "local staging cleared; free_GB=",\n              round(shutil.disk_usage(self.work).free / (1024 ** 3), 2))\n        return manifest\n\n\ndef connect_hf(token):\n    from huggingface_hub import HfApi\n    if not token or not token.strip():\n        raise RuntimeError("Thiếu Kaggle Secret HF_TOKEN2")\n    api = HfApi(token=token)\n    matches = []\n    for kind in ("dataset", "model"):\n        try:\n            api.repo_info(repo_id=HF_REPO_ID, repo_type=kind)\n            matches.append(kind)\n        except Exception:\n            pass\n    if len(matches) != 1:\n        raise RuntimeError("Không xác định duy nhất repo_type của " + HF_REPO_ID + "; kiểm tra token/quyền/repo")\n    return api, matches[0]\n', 'scripts/download_hf_checkpoints.py': '"""Download completed E3/E4 snapshots from the new HF repo, without loading torch.\n\nRun from the project root: python -m scripts.download_hf_checkpoints\nUses HF_TOKEN or the Hugging Face login cache; never modifies remote files.\n"""\n\nimport argparse\nimport json\nimport os\nimport re\nimport shutil\nfrom pathlib import Path\n\nfrom src.kaggle_hf_shared import atomic_json, sha256_file\n\n\nROOT = Path(__file__).resolve().parents[1]\nREPO_ID = "13ffa/deep"\nDATASET_HASH = "84a776e32c59063930b3e036d91876921163f48de70a9e85eb8c2b3cc2ddcf52"\nMODELS = {\n    "E3": ("YOLO 11n", "yolo11n.pt", "707ac9a6ed2eb1545007c8dd4ac752429731ca9b29cd585f8319c375d3de0d70"),\n    "E4": ("YOLO 11s", "yolo11s.pt", "684c8c281633c537094601c84bf3cdea5b674126230a69b74004380d6276aa70"),\n}\nREQUIRED = {"recovery_full.pt", "best.pt", "results.csv", "args.yaml", "config.json", "environment.json"}\n\n\ndef safe_filename(name):\n    if not isinstance(name, str) or not re.fullmatch(r"[A-Za-z0-9_][A-Za-z0-9_.-]*", name):\n        raise RuntimeError("Invalid artifact filename")\n    return name\n\n\ndef download_model(fetch, experiment, output, epochs, provenance):\n    """fetch reads only from the single commit selected by main()."""\n    folder, model, config_hash = MODELS[experiment]\n    pointer_path = fetch(folder + "/latest.json")\n    pointer = json.loads(pointer_path.read_text(encoding="utf-8"))\n    snapshot = pointer.get("snapshot")\n    prefix = folder + "/snapshots/"\n    if not isinstance(snapshot, str) or not snapshot.startswith(prefix):\n        raise RuntimeError("Invalid latest snapshot pointer")\n    snapshot_id = safe_filename(snapshot[len(prefix):])\n    manifest_path = fetch(snapshot + "/manifest.json")\n    if sha256_file(manifest_path) != pointer.get("manifest_sha256"):\n        raise RuntimeError("Manifest SHA256 mismatch")\n    manifest = json.loads(manifest_path.read_text(encoding="utf-8"))\n    if (manifest.get("schema") != 2 or manifest.get("snapshot") != snapshot\n            or manifest.get("folder") != folder or manifest.get("model") != model\n            or manifest.get("dataset_sha256") != DATASET_HASH\n            or manifest.get("config_sha256") != config_hash):\n        raise RuntimeError("Snapshot model/config/dataset identity mismatch")\n    if (type(manifest.get("completed_epoch")) is not int\n            or manifest["completed_epoch"] != epochs\n            or manifest.get("is_complete") is not True\n            or manifest.get("target_epochs", epochs) != epochs\n            or manifest.get("next_sample", 0) != 0):\n        raise RuntimeError(f"{experiment}: latest snapshot is not COMPLETE at epoch {epochs}")\n    hashes = manifest.get("sha256")\n    if not isinstance(hashes, dict) or not hashes:\n        raise RuntimeError("Missing artifact checksums")\n    for name, checksum in hashes.items():\n        safe_filename(name)\n        if name in {"manifest.json", "latest.json", "download_receipt.json"}:\n            raise RuntimeError("Reserved artifact filename")\n        if not isinstance(checksum, str) or not re.fullmatch(r"[0-9a-f]{64}", checksum):\n            raise RuntimeError("Invalid SHA256")\n    aliases = manifest.get("artifact_aliases", {})\n    if not isinstance(aliases, dict) or any(\n        alias != "best.pt" or target != "recovery_full.pt" or alias in hashes or target not in hashes\n        for alias, target in aliases.items()\n    ):\n        raise RuntimeError("Invalid best checkpoint alias")\n    if not REQUIRED.issubset(set(hashes) | set(aliases)):\n        raise RuntimeError("Snapshot missing checkpoint/metrics/config artifacts")\n\n    # Verify every artifact before publishing a usable local checkpoint directory.\n    downloaded = {}\n    for name, checksum in hashes.items():\n        path = fetch(snapshot + "/" + name)\n        if sha256_file(path) != checksum:\n            raise RuntimeError("Artifact SHA256 mismatch: " + name)\n        downloaded[name] = path\n    destination = Path(output) / experiment / f"epoch{epochs:03d}" / snapshot_id\n    destination.mkdir(parents=True, exist_ok=True)\n    for name, path in {**downloaded, "manifest.json": manifest_path, "latest.json": pointer_path}.items():\n        temp = destination / (name + ".tmp")\n        shutil.copyfile(path, temp)\n        os.replace(temp, destination / name)\n    for alias, target in aliases.items():\n        temp = destination / (alias + ".tmp")\n        shutil.copyfile(destination / target, temp)\n        os.replace(temp, destination / alias)\n    receipt = {**provenance, "experiment": experiment, "snapshot": snapshot,\n               "completed_epoch": epochs, "verified": True,\n               "best_sha256": sha256_file(destination / "best.pt"),\n               "manifest_sha256": pointer["manifest_sha256"]}\n    atomic_json(destination / "download_receipt.json", receipt)\n    print(f"{experiment}: COMPLETE {epochs}/{epochs}, SHA256 PASS: {destination}", flush=True)\n    return destination\n\n\ndef main(argv=None):\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--models", nargs="+", choices=MODELS, default=["E3", "E4"])\n    parser.add_argument("--repo-id", default=REPO_ID)\n    parser.add_argument("--repo-type", choices=("auto", "dataset", "model"), default="auto")\n    parser.add_argument("--epochs", type=int, choices=(10, 30), default=30)\n    parser.add_argument("--output", type=Path, default=ROOT / "artifacts/hf_checkpoints")\n    args = parser.parse_args(argv)\n    from huggingface_hub import HfApi, hf_hub_download\n    from huggingface_hub.errors import RepositoryNotFoundError\n\n    token = os.environ.get("HF_TOKEN") or None\n    api = HfApi(token=token)\n    kinds = ("dataset", "model") if args.repo_type == "auto" else (args.repo_type,)\n    for kind in kinds:\n        try:\n            info = api.repo_info(repo_id=args.repo_id, repo_type=kind)\n            break\n        except RepositoryNotFoundError:\n            if kind == kinds[-1]:\n                raise RuntimeError("Cannot access HF repo. Set HF_TOKEN with read access or use hf auth login.") from None\n    revision = info.sha\n    print(f"Read-only download: {args.repo_id} ({kind}), commit {revision}", flush=True)\n\n    def fetch(filename):\n        return Path(hf_hub_download(repo_id=args.repo_id, repo_type=kind, revision=revision,\n                                     filename=filename, token=token))\n\n    for experiment in dict.fromkeys(args.models):\n        download_model(fetch, experiment, args.output, args.epochs,\n                       {"repo_id": args.repo_id, "repo_type": kind, "revision": revision})\n\n\nif __name__ == "__main__":\n    main()\n', 'scripts/evaluate_30epoch_outputs.py': '"""Offline VALID calibration then locked TEST for the supplied E3/E4 Kaggle ZIPs."""\nfrom __future__ import annotations\n\nimport argparse\nimport csv\nimport io\nimport json\nimport os\nimport shutil\nimport time\nimport zipfile\nfrom dataclasses import asdict\nfrom pathlib import Path\n\nfrom src.kaggle_hf_shared import atomic_json, sha256_file\nfrom scripts.download_hf_checkpoints import DATASET_HASH, MODELS, REQUIRED, safe_filename\n\nROOT = Path(__file__).resolve().parents[1]\nCOUNTS = {"valid": 3318, "test": 1611}\nINFERENCE = {"conf": 0.01, "iou": 0.7, "max_det": 300, "imgsz": 640}\n\n\ndef prepare_checkpoint(source, experiment, output):\n    """Select artifacts from the verified latest snapshot, including compact aliases."""\n    folder, model, config = MODELS[experiment]\n    with zipfile.ZipFile(source) as archive:\n        names = archive.namelist()\n        if len(names) != len(set(names)):\n            raise RuntimeError("Duplicate ZIP entries")\n        manifests = [name for name in names if name.endswith("/last_hf_manifest.json")]\n        if len(manifests) != 1:\n            raise RuntimeError("Expected one latest training manifest in " + str(source))\n        manifest = json.loads(archive.read(manifests[0]))\n        if (manifest.get("schema") != 2 or manifest.get("folder") != folder\n                or manifest.get("model") != model or manifest.get("config_sha256") != config\n                or manifest.get("dataset_sha256") != DATASET_HASH\n                or type(manifest.get("completed_epoch")) is not int or manifest["completed_epoch"] != 30\n                or manifest.get("is_complete") is not True or manifest.get("target_epochs") != 30):\n            raise RuntimeError("Not the expected COMPLETE 30-epoch checkpoint: " + experiment)\n        snap = manifest["snapshot"]\n        prefix = folder + "/snapshots/"\n        if not snap.startswith(prefix):\n            raise RuntimeError("Invalid snapshot path")\n        safe_filename(snap[len(prefix):])\n        hashes = manifest["sha256"]\n        aliases = manifest.get("artifact_aliases", {})\n        if aliases and (aliases != {"best.pt": "recovery_full.pt"} or "best.pt" in hashes):\n            raise RuntimeError("Unexpected checkpoint aliases")\n        if not REQUIRED.issubset(set(hashes) | set(aliases)):\n            raise RuntimeError("Missing snapshot artifacts")\n        destination = Path(output) / experiment / "checkpoint"\n        destination.mkdir(parents=True, exist_ok=True)\n        for name, expected in hashes.items():\n            safe_filename(name)\n            # ZIP may also contain source snapshots and stripped native best.pt.\n            # Match only this manifest\'s immutable snapshot directory.\n            candidates = [entry for entry in names if entry.endswith("/" + snap + "/" + name)]\n            if not candidates:\n                raise RuntimeError("Missing snapshot artifact in ZIP: " + name)\n            path = destination / name\n            if path.is_file() and sha256_file(path) == expected:\n                continue\n            temp = destination / (name + ".tmp")\n            with archive.open(candidates[0]) as src, temp.open("wb") as dest:\n                shutil.copyfileobj(src, dest)\n            if sha256_file(temp) != expected:\n                raise RuntimeError("ZIP artifact SHA256 mismatch: " + name)\n            os.replace(temp, path)\n        if aliases:\n            target = destination / "best.pt"\n            expected = hashes["recovery_full.pt"]\n            if not target.exists() or sha256_file(target) != expected:\n                shutil.copyfile(destination / "recovery_full.pt", target)\n        rows = [{key.strip(): value.strip() for key, value in row.items()} for row in\n                csv.DictReader(io.StringIO((destination / "results.csv").read_text(encoding="utf-8-sig")))]\n        epochs = [int(float(row["epoch"])) for row in rows]\n        if epochs != list(range(1, 31)):\n            raise RuntimeError("Training CSV must contain contiguous epochs 1..30")\n        atomic_json(destination / "manifest.json", manifest)\n        audit = {"model": experiment, "archive": str(Path(source).resolve()),\n                 "archive_sha256": sha256_file(source), "completed_epoch": 30,\n                 "snapshot": snap, "best_sha256": sha256_file(destination / "best.pt"),\n                 "dataset_sha256": DATASET_HASH, "training_last_row": rows[-1],\n                 "best_role": "recovery_full.pt alias" if aliases else "best.pt"}\n        atomic_json(destination / "audit.json", audit)\n        print(f"{experiment}: verified checkpoint 30/30; best SHA256={audit[\'best_sha256\']}", flush=True)\n        return destination / "best.pt", audit\n\n\ndef read_cache(path):\n    """Recover only a torn final JSONL write; reject corruption in earlier rows."""\n    if not path.exists():\n        return []\n    records = []\n    with path.open("rb+") as stream:\n        while True:\n            position = stream.tell()\n            line = stream.readline()\n            if not line:\n                break\n            try:\n                records.append(json.loads(line))\n                if not line.endswith(b"\\n"):\n                    stream.write(b"\\n")\n            except (ValueError, UnicodeDecodeError):\n                if stream.read(1):\n                    raise RuntimeError("Corrupt prediction cache: " + str(path)) from None\n                stream.truncate(position)\n                print("Recovered interrupted final cache line:", path, flush=True)\n                break\n    return records\n\n\ndef collect(model, experiment, split, dataset, output, metadata, batch_size, device):\n    from app.calibrate_confidence import ground_truth\n    images, labels = dataset / split / "images", dataset / split / "labels"\n    paths = sorted(p for p in images.iterdir() if p.suffix.lower() in {".jpg", ".jpeg", ".png"})\n    if len(paths) != COUNTS[split] or any(not (labels / (p.stem + ".txt")).is_file() for p in paths):\n        raise RuntimeError(f"{split}: expected {COUNTS[split]} images and matching labels")\n    directory = output / experiment / split\n    directory.mkdir(parents=True, exist_ok=True)\n    cache, meta_path = directory / "predictions.jsonl", directory / "metadata.json"\n    expected = {**metadata, "model": experiment, "split": split, "images": len(paths),\n                "inference": INFERENCE}\n    if meta_path.exists():\n        if json.loads(meta_path.read_text(encoding="utf-8")) != expected:\n            raise RuntimeError("Prediction cache provenance mismatch: " + str(directory))\n    elif cache.exists():\n        raise RuntimeError("Prediction cache lacks metadata")\n    else:\n        atomic_json(meta_path, expected)\n    records = read_cache(cache)\n    if len(records) > len(paths) or any(row["image"] != path.name for row, path in zip(records, paths)):\n        raise RuntimeError("Cached image order mismatch")\n    print(f"{experiment} {split.upper()}: resume {len(records)}/{len(paths)}", flush=True)\n    started = last_log = time.monotonic()\n    initial_count = len(records)\n    with cache.open("a", encoding="utf-8") as stream:\n        for offset in range(len(records), len(paths), batch_size):\n            batch = paths[offset:offset + batch_size]\n            results = model.predict(source=[str(p) for p in batch], **INFERENCE,\n                                    device=device, batch=batch_size, verbose=False, save=False)\n            if len(results) != len(batch):\n                raise RuntimeError("Prediction count mismatch")\n            for path, result in zip(batch, results, strict=True):\n                if Path(result.path).name != path.name:\n                    raise RuntimeError("Prediction image mismatch")\n                height, width = result.orig_shape\n                boxes = result.boxes\n                predictions = [] if boxes is None else [\n                    [int(cls), float(score), *map(float, box)]\n                    for cls, score, box in zip(boxes.cls.cpu().tolist(), boxes.conf.cpu().tolist(),\n                                               boxes.xyxy.cpu().tolist(), strict=True)]\n                row = {"image": path.name, "width": width, "height": height,\n                       "gt": ground_truth(labels / (path.stem + ".txt"), width, height), "pred": predictions}\n                stream.write(json.dumps(row, ensure_ascii=False) + "\\n")\n                stream.flush()\n                records.append(row)\n            now = time.monotonic()\n            if offset == initial_count or len(records) % 100 < batch_size or now - last_log >= 60 or len(records) == len(paths):\n                elapsed = max(now - started, 1e-6)\n                rate = (len(records) - initial_count) / elapsed\n                eta = (len(paths) - len(records)) / max(rate, 1e-6)\n                print(f"{experiment} {split.upper()} {len(records)}/{len(paths)} "\n                      f"({100 * len(records) / len(paths):.1f}%) elapsed={elapsed:.0f}s ETA={eta:.0f}s", flush=True)\n                last_log = now\n    return records, cache\n\n\ndef save_metrics(records, threshold, output, provenance, names):\n    from app.evaluator import evaluate_records\n    print("Computing mAP:", output, flush=True)\n    metrics = evaluate_records(records, threshold)\n    metrics.update(provenance)\n    metrics["f2"] = (5 * metrics["precision"] * metrics["recall"] /\n                     (4 * metrics["precision"] + metrics["recall"])) if metrics["precision"] or metrics["recall"] else 0.0\n    atomic_json(output / "metrics.json", metrics)\n    with (output / "per_class.csv").open("w", newline="", encoding="utf-8-sig") as stream:\n        writer = csv.DictWriter(stream, fieldnames=["class_id", "class_name", "gt_count", "ap50", "ap50_95", "tp", "fp", "fn"])\n        writer.writeheader()\n        writer.writerows({**row, "class_name": names[row["class_id"]]} for row in metrics["per_class"])\n    print(f"{provenance[\'model\']} {provenance[\'split\'].upper()} mAP50={metrics[\'map50\']:.6f} "\n          f"mAP50-95={metrics[\'map50_95\']:.6f} P={metrics[\'precision\']:.6f} R={metrics[\'recall\']:.6f}", flush=True)\n    return metrics\n\n\ndef main(argv=None):\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--e3-zip", type=Path, default=ROOT / "results_zips" / "results_e3.zip")\n    parser.add_argument("--e4-zip", type=Path, default=ROOT / "results_zips" / "results_e4.zip")\n    parser.add_argument("--models", nargs="+", choices=MODELS, default=["E3", "E4"])\n    parser.add_argument("--dataset", type=Path, default=ROOT / "data/processed/rod_detection_final")\n    parser.add_argument("--output", type=Path, default=ROOT / "results/30epochs")\n    parser.add_argument("--batch", type=int, default=4)\n    parser.add_argument("--device", default="auto", help="auto, cpu or GPU index such as 0")\n    args = parser.parse_args(argv)\n    if args.batch < 1:\n        parser.error("--batch must be positive")\n    os.environ["YOLO_AUTOINSTALL"] = "false"\n    os.environ["HF_HUB_OFFLINE"] = "1"\n    os.environ["YOLO_OFFLINE"] = "true"\n    import torch\n    import ultralytics\n    import yaml\n    from ultralytics import YOLO\n    from app.calibrate_confidence import choose_threshold, score_predictions\n    from app.evaluator import METRIC_PROTOCOL\n\n    if ultralytics.__version__ != "8.4.165":\n        raise RuntimeError("Use ultralytics==8.4.165 to match these training checkpoints")\n    device = ("0" if torch.cuda.is_available() else "cpu") if args.device == "auto" else args.device\n    print(f"Device={device}; torch={torch.__version__}; ultralytics={ultralytics.__version__}", flush=True)\n    if device != "cpu" and not torch.cuda.is_available():\n        raise RuntimeError("CUDA requested but this PyTorch installation has no CUDA support")\n    yaml_path = args.dataset / "data.yaml"\n    if sha256_file(yaml_path) != DATASET_HASH:\n        raise RuntimeError("Dataset YAML differs from the frozen training dataset")\n    config = yaml.safe_load(yaml_path.read_text(encoding="utf-8"))\n    raw_names = config["names"]\n    names = {int(k): v for k, v in raw_names.items()} if isinstance(raw_names, dict) else dict(enumerate(raw_names))\n    if config.get("nc") != 25 or set(names) != set(range(25)):\n        raise RuntimeError("Expected 25 ROD classes")\n    models = list(dict.fromkeys(args.models))\n    prepared, summaries = {}, {}\n    for experiment in models:\n        checkpoint, audit = prepare_checkpoint(args.e3_zip if experiment == "E3" else args.e4_zip,\n                                               experiment, args.output)\n        metadata = {"checkpoint_sha256": audit["best_sha256"], "data_yaml_sha256": DATASET_HASH,\n                    "ultralytics": ultralytics.__version__, "torch": torch.__version__, "device": device}\n        model = YOLO(str(checkpoint))\n        actual = {int(k): v for k, v in model.names.items()}\n        if actual != names or model.task != "detect" or model.model.yaml.get("scale") != ("n" if experiment == "E3" else "s"):\n            raise RuntimeError("Checkpoint architecture or class mapping mismatch")\n        records, cache = collect(model, experiment, "valid", args.dataset, args.output, metadata, args.batch, device)\n        print(f"{experiment}: selecting threshold from VALID only (micro F2)...", flush=True)\n        scores = score_predictions(records)\n        selected = choose_threshold(scores)\n        valid_dir = args.output / experiment / "valid"\n        with (valid_dir / "threshold_sweep.csv").open("w", newline="", encoding="utf-8") as stream:\n            writer = csv.DictWriter(stream, fieldnames=list(asdict(selected)))\n            writer.writeheader()\n            writer.writerows(asdict(score) for score in scores)\n        evidence = {"model": experiment, "split": "valid", "checkpoint_sha256": audit["best_sha256"],\n                    "prediction_sha256": sha256_file(cache), "status": "FULL_30EPOCH"}\n        summaries[experiment] = {"valid": save_metrics(records, selected.threshold, valid_dir, evidence, names)}\n        lock = {**metadata, "model": experiment, "completed_epoch": 30,\n                "threshold": selected.threshold, "threshold_selection": "VALID micro-F2; ties precision then threshold",\n                "valid_prediction_sha256": sha256_file(cache), "inference": INFERENCE,\n                "metric_protocol": METRIC_PROTOCOL}\n        lock_path = args.output / experiment / "evaluation_lock.json"\n        if lock_path.exists() and json.loads(lock_path.read_text(encoding="utf-8")) != lock:\n            raise RuntimeError("Existing 30-epoch TEST lock differs; refusing overwrite")\n        atomic_json(lock_path, lock)\n        prepared[experiment] = (checkpoint, metadata, lock, sha256_file(lock_path))\n        del model, records\n    # Complete and lock all requested VALID runs before opening TEST images.\n    for experiment, (checkpoint, metadata, lock, lock_sha) in prepared.items():\n        lock_path = args.output / experiment / "evaluation_lock.json"\n        if sha256_file(lock_path) != lock_sha or sha256_file(checkpoint) != metadata["checkpoint_sha256"]:\n            raise RuntimeError("Checkpoint/lock changed before TEST")\n        valid_cache = args.output / experiment / "valid/predictions.jsonl"\n        if sha256_file(valid_cache) != lock["valid_prediction_sha256"]:\n            raise RuntimeError("VALID evidence changed before TEST")\n        model = YOLO(str(checkpoint))\n        records, cache = collect(model, experiment, "test", args.dataset, args.output,\n                                 {**metadata, "lock_sha256": lock_sha}, args.batch, device)\n        evidence = {"model": experiment, "split": "test", "checkpoint_sha256": metadata["checkpoint_sha256"],\n                    "lock_sha256": lock_sha, "prediction_sha256": sha256_file(cache), "status": "FINAL_LOCKED_30EPOCH"}\n        summaries[experiment]["test"] = save_metrics(records, lock["threshold"], args.output / experiment / "test", evidence, names)\n        del model, records\n    summary_path = args.output / ("summary_" + "_".join(models) + ".json")\n    atomic_json(summary_path, summaries)\n    with summary_path.with_suffix(".csv").open("w", newline="", encoding="utf-8-sig") as stream:\n        fields = ["model", "split", "images", "operating_threshold", "map50", "map50_95", "precision", "recall", "f2"]\n        writer = csv.DictWriter(stream, fieldnames=fields, extrasaction="ignore")\n        writer.writeheader()\n        writer.writerows(metrics for split in summaries.values() for metrics in split.values())\n    print("DONE:", summary_path, flush=True)\n\n\nif __name__ == "__main__":\n    main()\n', 'app/evaluator.py': '"""Shared ROD detection metrics for all three model adapters.\n\nProtocol: class-aware, one-to-one IoU matching; 101-point interpolated AP at\nIoU .50:.05:.95; predictions collected at score >= .01, then max 100 scored\ndetections per image for AP and max 300 for operating Precision/Recall (the\nsame collection cap used during threshold calibration). ROD has no crowd/ignore\nannotations, so these are COCO-style all-area metrics, not a pycocotools run.\n"""\n\nfrom __future__ import annotations\n\nfrom dataclasses import dataclass\n\nIOU_THRESHOLDS = tuple(round(0.5 + 0.05 * i, 2) for i in range(10))\nMAX_DET = 100\nOPERATING_MAX_DET = 300\nMETRIC_PROTOCOL = {\n    "iou_thresholds": list(IOU_THRESHOLDS), "ap_recall_points": 101,\n    "max_detections_per_image": MAX_DET,\n    "operating_max_detections_per_image": OPERATING_MAX_DET,\n    "match": "class-aware greedy, score descending",\n    "area": "all", "crowd_ignore": False,\n    "note": "COCO-style interpolation; ROD has no crowd/ignore labels",\n}\n\n\ndef iou(a: list[float], b: list[float]) -> float:\n    x1, y1 = max(a[0], b[0]), max(a[1], b[1])\n    x2, y2 = min(a[2], b[2]), min(a[3], b[3])\n    inter = max(0.0, x2 - x1) * max(0.0, y2 - y1)\n    area_a = max(0.0, a[2] - a[0]) * max(0.0, a[3] - a[1])\n    area_b = max(0.0, b[2] - b[0]) * max(0.0, b[3] - b[1])\n    union = area_a + area_b - inter\n    return inter / union if union > 0 else 0.0\n\n\n@dataclass(frozen=True)\nclass ClassMetrics:\n    class_id: int\n    gt_count: int\n    ap50: float | None\n    ap50_95: float | None\n    tp: int\n    fp: int\n    fn: int\n\n\ndef _validate(records: list[dict]) -> None:\n    if len({row["image"] for row in records}) != len(records):\n        raise ValueError("Duplicate image in evaluation records")\n    for row in records:\n        width, height = row["width"], row["height"]\n        if width <= 0 or height <= 0:\n            raise ValueError("Invalid image dimensions")\n        for kind in ("gt", "pred"):\n            for item in row[kind]:\n                if len(item) != (5 if kind == "gt" else 6):\n                    raise ValueError("Invalid annotation/prediction row")\n                cls = item[0]\n                if isinstance(cls, bool) or int(cls) != cls or not 0 <= cls < 25:\n                    raise ValueError("Class ID outside ROD 0..24")\n                box = item[1:] if kind == "gt" else item[2:]\n                # Float32 model/export rounding can put an edge microscopically\n                # outside the image (for example 999.000061 on a 999 px image).\n                # Keep the tolerance far below one pixel; genuinely out-of-bounds\n                # boxes are still rejected and zero-area predictions remain FP.\n                tolerance = 1e-3\n                order = (box[0] < box[2] and box[1] < box[3]) if kind == "gt" else (\n                    box[0] <= box[2] and box[1] <= box[3])\n                if not (order and -tolerance <= box[0] <= width + tolerance and\n                        -tolerance <= box[1] <= height + tolerance and\n                        -tolerance <= box[2] <= width + tolerance and\n                        -tolerance <= box[3] <= height + tolerance):\n                    raise ValueError("Invalid pixel xyxy box")\n                if kind == "pred" and not 0 <= item[1] <= 1:\n                    raise ValueError("Invalid confidence")\n\n\ndef _class_outcomes(records: list[dict], class_id: int, threshold: float,\n                    score_cutoff: float = 0.0,\n                    max_det: int = MAX_DET) -> tuple[list[tuple[float, bool]], int]:\n    ground_truth: dict[str, list[list[float]]] = {}\n    predictions: list[tuple[float, str, list[float]]] = []\n    for row in records:\n        image = row["image"]\n        ground_truth[image] = [item[1:] for item in row["gt"] if item[0] == class_id]\n        selected = sorted(row["pred"], key=lambda item: -item[1])[:max_det]\n        predictions.extend((float(item[1]), image, item[2:]) for item in selected\n                           if item[0] == class_id and item[1] >= score_cutoff)\n    predictions.sort(key=lambda item: -item[0])\n    used: dict[str, set[int]] = {image: set() for image in ground_truth}\n    outcomes = []\n    for score, image, box in predictions:\n        candidates = [(iou(box, gt), index) for index, gt in enumerate(ground_truth[image])\n                      if index not in used[image]]\n        best, index = max(candidates, default=(0.0, -1))\n        hit = best >= threshold\n        if hit:\n            used[image].add(index)\n        outcomes.append((score, hit))\n    return outcomes, sum(map(len, ground_truth.values()))\n\n\ndef _average_precision(outcomes: list[tuple[float, bool]], gt_count: int) -> float | None:\n    if gt_count == 0:\n        return None\n    true, false = 0, 0\n    curve: list[tuple[float, float]] = []\n    for _, hit in outcomes:\n        true += int(hit)\n        false += int(not hit)\n        curve.append((true / gt_count, true / (true + false)))\n    return sum(max((precision for recall, precision in curve if recall >= r), default=0.0)\n               for r in (i / 100 for i in range(101))) / 101\n\n\ndef evaluate_records(records: list[dict], operating_threshold: float) -> dict:\n    """Return identical metric definitions for E3, E4 and E6 records."""\n    if not 0 < operating_threshold < 1:\n        raise ValueError("Operating threshold must be in (0,1)")\n    _validate(records)\n    per_class = []\n    for class_id in range(25):\n        aps = []\n        for match_iou in IOU_THRESHOLDS:\n            outcomes, count = _class_outcomes(records, class_id, match_iou)\n            aps.append(_average_precision(outcomes, count))\n        selected, count = _class_outcomes(records, class_id, 0.5, operating_threshold,\n                                           max_det=OPERATING_MAX_DET)\n        tp = sum(hit for _, hit in selected)\n        fp = len(selected) - tp\n        per_class.append(ClassMetrics(\n            class_id, count, aps[0],\n            sum(aps) / len(aps) if count else None,\n            tp, fp, count - tp,\n        ))\n    tp = sum(item.tp for item in per_class)\n    fp = sum(item.fp for item in per_class)\n    fn = sum(item.fn for item in per_class)\n    active = [item for item in per_class if item.gt_count]\n    return {\n        "images": len(records), "ground_truth_objects": sum(item.gt_count for item in per_class),\n        "operating_threshold": operating_threshold,\n        "precision": tp / (tp + fp) if tp + fp else 0.0,\n        "recall": tp / (tp + fn) if tp + fn else 0.0,\n        "tp": tp, "fp": fp, "fn": fn,\n        "map50": sum(item.ap50 for item in active) / len(active) if active else None,\n        "map50_95": sum(item.ap50_95 for item in active) / len(active) if active else None,\n        "per_class": [item.__dict__ for item in per_class],\n        "protocol": METRIC_PROTOCOL,\n    }\n', 'app/calibrate_confidence.py': 'from __future__ import annotations\nfrom dataclasses import dataclass\nfrom pathlib import Path\nMATCH_IOU = 0.5\nTHRESHOLDS = [round(i / 100, 2) for i in range(1, 96)]\n\n@dataclass(frozen=True)\nclass Score:\n    threshold: float\n    tp: int\n    fp: int\n    fn: int\n    precision: float\n    recall: float\n    f2: float\n\ndef ground_truth(label_path: Path, width: int, height: int) -> list[list[float]]:\n    if not label_path.is_file():\n        raise RuntimeError(f"Thiếu nhãn VALID: {label_path}")\n    rows = []\n    for line in label_path.read_text(encoding="utf-8").splitlines():\n        if not line.strip():\n            continue\n        values = [float(value) for value in line.split()]\n        if len(values) != 5 or int(values[0]) != values[0] or not 0 <= values[0] < 25:\n            raise RuntimeError(f"Nhãn YOLO không hợp lệ: {label_path}")\n        class_id, x, y, w, h = values\n        if not (0 <= x <= 1 and 0 <= y <= 1 and 0 < w <= 1 and 0 < h <= 1):\n            raise RuntimeError(f"Bbox YOLO không hợp lệ: {label_path}")\n        rows.append([int(class_id), (x - w / 2) * width, (y - h / 2) * height,\n                     (x + w / 2) * width, (y + h / 2) * height])\n    return rows\n\ndef intersection_over_union(a: list[float], b: list[float]) -> float:\n    left, top = max(a[0], b[0]), max(a[1], b[1])\n    right, bottom = min(a[2], b[2]), min(a[3], b[3])\n    intersection = max(0.0, right - left) * max(0.0, bottom - top)\n    area_a = max(0.0, a[2] - a[0]) * max(0.0, a[3] - a[1])\n    area_b = max(0.0, b[2] - b[0]) * max(0.0, b[3] - b[1])\n    union = area_a + area_b - intersection\n    return intersection / union if union > 0 else 0.0\n\ndef score_predictions(records: list[dict], thresholds: list[float] = THRESHOLDS) -> list[Score]:\n    """Match each class-correct prediction to at most one GT at IoU >= 0.5."""\n    outcomes: list[tuple[float, bool]] = []\n    total_gt = 0\n    for record in records:\n        gt = record["gt"]\n        total_gt += len(gt)\n        used: set[int] = set()\n        for pred in sorted(record["pred"], key=lambda row: -row[1]):\n            cls, conf, *box = pred\n            eligible = [\n                (intersection_over_union(box, item[1:]), index)\n                for index, item in enumerate(gt)\n                if index not in used and item[0] == cls\n            ]\n            best_iou, best_index = max(eligible, default=(0.0, -1))\n            matched = best_iou >= MATCH_IOU\n            if matched:\n                used.add(best_index)\n            outcomes.append((conf, matched))\n    scores = []\n    for threshold in thresholds:\n        selected = [matched for confidence, matched in outcomes if confidence >= threshold]\n        tp = sum(selected)\n        fp = len(selected) - tp\n        fn = total_gt - tp\n        precision = tp / (tp + fp) if tp + fp else 0.0\n        recall = tp / total_gt if total_gt else 0.0\n        f2 = 5 * precision * recall / (4 * precision + recall) if precision or recall else 0.0\n        scores.append(Score(threshold, tp, fp, fn, precision, recall, f2))\n    return scores\n\ndef choose_threshold(scores: list[Score]) -> Score:\n    if not scores:\n        raise RuntimeError("Không có ngưỡng để đánh giá.")\n    # F2 weights recall four times as strongly as precision; tie favors fewer FP.\n    return max(scores, key=lambda score: (score.f2, score.precision, score.threshold))', 'src/__init__.py': '', 'app/__init__.py': '', 'scripts/__init__.py': ''}
for name, content in SOURCES.items():
    path = CODE_ROOT / name
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(content, encoding='utf-8')
sys.path.insert(0, str(CODE_ROOT))
print('Standalone evaluation code ready', flush=True)


In [ ]:
from pathlib import Path
import json
import zipfile

INPUT = Path('/kaggle/input')
OUTPUT = Path('/kaggle/working/evaluation_30epochs')
# Optional: fill these when there are multiple matching inputs.
E3_ZIP = None
E4_ZIP = None
DATASET_ROOT = None
BATCH_SIZE = 4  # Reduce to 2 if GPU memory runs out.

def find_output(experiment, explicit):
    if explicit:
        return Path(explicit)
    zipped = list(INPUT.rglob('results_' + experiment.lower() + '.zip'))
    if len(zipped) == 1:
        return zipped[0]
    if len(zipped) > 1:
        raise RuntimeError('Multiple ZIPs found; set ' + experiment + '_ZIP explicitly')
    # Kaggle often exposes uploaded ZIP contents as already extracted files.
    expected_model = 'yolo11n.pt' if experiment == 'E3' else 'yolo11s.pt'
    matches = []
    for path in INPUT.rglob('last_hf_manifest.json'):
        manifest = json.loads(path.read_text(encoding='utf-8'))
        if manifest.get('model') == expected_model and manifest.get('completed_epoch') == 30:
            matches.append((path, manifest))
    if len(matches) != 1:
        raise RuntimeError('Attach the completed ' + experiment + ' Output ZIP as a Kaggle input dataset; found ' + str(len(matches)) + ' matching extracted outputs')
    path, manifest = matches[0]
    run = path.parent
    snapshot = manifest['snapshot']
    # Repack only this snapshot; no older checkpoints or HF cache needed.
    from scripts.download_hf_checkpoints import safe_filename
    folder = 'YOLO 11n' if experiment == 'E3' else 'YOLO 11s'
    prefix = folder + '/snapshots/'
    if not snapshot.startswith(prefix):
        raise RuntimeError('Invalid snapshot pointer')
    safe_filename(snapshot[len(prefix):])
    staged = Path('/kaggle/working/input_staging')
    staged.mkdir(parents=True, exist_ok=True)
    target = staged / ('results_' + experiment.lower() + '.zip')
    with zipfile.ZipFile(target, 'w', compression=zipfile.ZIP_STORED) as archive:
        archive.write(path, 'run/last_hf_manifest.json')
        for name in manifest['sha256']:
            safe_filename(name)
            suffix = snapshot + '/' + name
            candidates = [p for p in run.rglob(name) if p.as_posix().endswith('/' + suffix)]
            if len(candidates) != 1:
                raise RuntimeError('Missing or ambiguous snapshot artifact: ' + suffix)
            archive.write(candidates[0], 'run/hf_backup/download/' + suffix)
    return target

if DATASET_ROOT:
    dataset = Path(DATASET_ROOT)
else:
    datasets = [p.parent for p in INPUT.rglob('data.yaml')
                if (p.parent / 'valid/images').is_dir() and (p.parent / 'test/images').is_dir()]
    if len(datasets) != 1:
        raise RuntimeError('Attach the original ROD Final A4 dataset with data.yaml, valid/images, valid/labels, test/images, test/labels. If multiple matches exist, set DATASET_ROOT.')
    dataset = datasets[0]
e3_zip, e4_zip = find_output('E3', E3_ZIP), find_output('E4', E4_ZIP)
print('E3:', e3_zip, '\nE4:', e4_zip, '\nDataset:', dataset, '\nOutput:', OUTPUT, flush=True)


In [ ]:
from scripts.evaluate_30epoch_outputs import main
main(['--e3-zip', str(e3_zip), '--e4-zip', str(e4_zip), '--dataset', str(dataset),
      '--output', str(OUTPUT), '--batch', str(BATCH_SIZE), '--device', '0'])


In [ ]:
import shutil
archive = shutil.make_archive('/kaggle/working/evaluation_30epochs_results', 'zip', OUTPUT)
print('Download:', archive, flush=True)
print('Summary:', OUTPUT / 'summary_E3_E4.csv', flush=True)
